# 🛡️ Composite Hallucination Detection & Risk Assessment POC

**Objective:** Build a robust, multi-signal RAG risk scoring POC. We evaluate:
1. **Relevance Check (all-MiniLM-L6-v2)** — Cosine similarity between query and retrieved docs.
2. **Faithfulness Check (MiniCheck-Flan-T5-Large)** — Claims-level factuality assessment.
3. **Citation Coverage** — Measuring what percentage of retrieved docs were actually utilized.
4. **Composite Risk Score** — High risk defined as `(relevance is high) AND (faithfulness is low)`.
5. **Escalation Tier** — Borderline MiniCheck cases (scores 0.3 to 0.7) are escalated to an LLM-as-Judge.

**Datasets:**
- Hand-built adjacent-capability mock dataset (25 support bot examples)
- RAGTruth QA subset (subset of 30 examples from `wandb/RAGTruth-processed`)

---

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
import subprocess, sys

packages = [
    'torch',
    'lettucedetect',
    'sentence-transformers',
    'transformers',
    'sentencepiece',
    'pandas',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'requests',
    'datasets',
    'nltk',
]

print('Installing packages...')
for pkg in packages:
    try:
        subprocess.check_call(
            [sys.executable, '-m', 'pip', 'install', '-q', pkg],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print(f'  \u2705 {pkg}')
    except Exception as e:
        print(f'  \u26a0\ufe0f {pkg} - install issue: {e}')

print('\n\u2705 Dependency installation complete!')


In [ ]:
# ============================================================
# \U0001f527 Imports & NLTK Setup
# ============================================================
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import time
import re
import warnings
import nltk
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix

nltk.download('punkt', quiet=True)
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('ggplot')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')


---
## \U0001f4cb Section 1: Datasets Loading
We load our mock adjacent-capability dataset and slice the RAGTruth QA subset.

In [ ]:
# ============================================================
# Loading Datasets
# ============================================================
from datasets import load_dataset

# 1. Custom Mock Dataset (25 examples)
mock_dataset = [
    # === HALLUCINATED EXAMPLES (13) ===
    {
        "id": "H01",
        "query": "Can I push/pull code via GitHub integration in Site24x7?",
        "retrieved_docs": [
            "Site24x7 GitHub Integration: Monitor your GitHub repositories health, "
            "track commit frequency, pull request metrics, and repository activity. "
            "Set up alerts for unusual patterns in your codebase activity. Configure "
            "the integration from Admin > Third Party Integration > GitHub."
        ],
        "response": (
            "Yes, Site24x7 supports full GitHub push/pull integration. You can push "
            "and pull code directly from the Site24x7 dashboard through the GitHub "
            "integration available under Admin > Third Party Integration > GitHub."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H02",
        "query": "Does Zoho CRM support video calling with customers?",
        "retrieved_docs": [
            "Zoho CRM Telephony Integration: Make and receive calls directly within "
            "Zoho CRM using built-in telephony features. Log call details automatically, "
            "record calls for quality assurance, and associate calls with specific leads "
            "or contacts. Supports integration with major telephony providers like Twilio "
            "and RingCentral."
        ],
        "response": (
            "Yes, Zoho CRM supports video calling with customers through its built-in "
            "telephony integration. You can make video calls directly from lead or contact "
            "records using the telephony feature under Settings > Telephony."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H03",
        "query": "Can Zoho Desk automatically translate support tickets?",
        "retrieved_docs": [
            "Zoho Desk Multi-language Support: Zoho Desk supports multiple languages for "
            "your help center portal. Customers can view the help center in their preferred "
            "language. You can create knowledge base articles in multiple languages and "
            "configure language-specific email templates. Supported languages include English, "
            "Spanish, French, German, Japanese, and 40+ others."
        ],
        "response": (
            "Yes, Zoho Desk can automatically translate support tickets into different "
            "languages. When a ticket arrives, the system detects the language and translates "
            "it for your agents. Configure this under Setup > General Settings > Multi-language."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H04",
        "query": "Does Zoho Books support cryptocurrency payments?",
        "retrieved_docs": [
            "Zoho Books Multi-Currency Support: Handle transactions in multiple currencies "
            "with automatic exchange rate updates. Zoho Books supports 170+ currencies and "
            "automatically fetches real-time exchange rates. Record transactions, generate "
            "invoices, and manage accounts payable/receivable in foreign currencies."
        ],
        "response": (
            "Yes, Zoho Books supports cryptocurrency payments. You can accept Bitcoin, "
            "Ethereum, and other major cryptocurrencies through the multi-currency payment "
            "gateway. Configure crypto payments under Settings > Online Payments."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H05",
        "query": "Can Zoho Projects automatically assign tasks using AI?",
        "retrieved_docs": [
            "Zoho Projects Task Management: Create and organize tasks with customizable "
            "fields, priorities, and deadlines. Assign tasks to team members manually or "
            "in bulk. Use task dependencies to establish project workflows. Track task "
            "progress with Kanban boards, Gantt charts, and custom views."
        ],
        "response": (
            "Yes, Zoho Projects includes AI-powered automatic task assignment. The system "
            "analyzes team members workload, skills, and availability to automatically assign "
            "incoming tasks to the most suitable team member."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H06",
        "query": "Does Zoho Analytics support real-time streaming data ingestion?",
        "retrieved_docs": [
            "Zoho Analytics Data Import: Import data from various sources including files "
            "(CSV, Excel, JSON), databases (MySQL, PostgreSQL, SQL Server, Oracle), cloud "
            "storage (Google Drive, Dropbox, Box), and business applications. Schedule "
            "automatic data syncs at intervals ranging from every 30 minutes to daily/weekly."
        ],
        "response": (
            "Yes, Zoho Analytics supports real-time streaming data ingestion. You can set up "
            "live data streams that push data continuously into your analytics workspace. "
            "Configure streaming endpoints under Data Sources > Real-time Streams."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H07",
        "query": "Can Zoho Mail compose AI-generated replies?",
        "retrieved_docs": [
            "Zoho Mail Smart Features: Use canned responses to quickly reply to common "
            "emails with pre-written templates. Create and manage response templates "
            "organized by categories. Set up auto-reply (out-of-office) messages with "
            "customizable schedules. Use email filters and rules to organize incoming mail."
        ],
        "response": (
            "Yes, Zoho Mail can compose AI-generated replies based on email context. The "
            "AI analyzes the incoming email and generates contextual reply suggestions. You "
            "can find this feature under the Smart Reply option when composing a response."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H08",
        "query": "Does Site24x7 support fully automated server self-healing?",
        "retrieved_docs": [
            "Site24x7 IT Automation: Configure automated actions triggered by monitor "
            "alerts. Supported automation actions include restarting Windows services, "
            "executing custom scripts on servers, sending webhook notifications, creating "
            "tickets in ITSM tools, and triggering AWS Lambda functions."
        ],
        "response": (
            "Yes, Site24x7 supports fully automated server self-healing. When issues are "
            "detected, the system automatically diagnoses the root cause and applies the "
            "appropriate fix without human intervention, including automatic scaling, "
            "configuration rollback, and self-healing procedures."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H09",
        "query": "Can Zoho Creator deploy apps to App Store/Play Store?",
        "retrieved_docs": [
            "Zoho Creator Mobile Access: Access your Zoho Creator applications on mobile "
            "devices through the Zoho Creator mobile app. View and manage records, submit "
            "forms, and receive push notifications on iOS and Android. Customize the mobile "
            "layout of your forms and reports for a better mobile experience."
        ],
        "response": (
            "Yes, Zoho Creator allows you to deploy your applications as standalone mobile "
            "apps on the Apple App Store and Google Play Store. You can customize the app "
            "icon, splash screen, and branding via Deploy > App Store Publishing."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H10",
        "query": "Does Zoho Desk support voice-based ticket creation with speech-to-text?",
        "retrieved_docs": [
            "Zoho Desk Multi-Channel Ticketing: Create tickets from multiple channels "
            "including email, web forms, social media (Twitter, Facebook), live chat, phone "
            "calls, and community forums. Each channel is automatically mapped to a "
            "department. Phone call tickets are created when agents log call details."
        ],
        "response": (
            "Yes, Zoho Desk supports voice-based ticket creation with built-in speech-to-text "
            "AI. Customers can call in and the system automatically transcribes the conversation "
            "in real-time, creating a ticket with the full transcript and auto-detected category."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H11",
        "query": "Can Zoho Books automatically categorize expenses using AI?",
        "retrieved_docs": [
            "Zoho Books Expense Tracking: Record and manage business expenses. Attach "
            "receipts by uploading images or forwarding email receipts. Create expense "
            "categories and tags for organization. Set up recurring expenses for regular "
            "payments. Generate expense reports by category, vendor, date range, or project."
        ],
        "response": (
            "Yes, Zoho Books uses AI to automatically categorize your expenses. When you "
            "upload a receipt, the AI engine reads the receipt, extracts vendor and amount "
            "information, and automatically assigns the expense to the correct category."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H12",
        "query": "Can Zoho Analytics generate natural language insights from charts?",
        "retrieved_docs": [
            "Zoho Analytics Visualization: Create interactive charts, pivot tables, and "
            "dashboards. Choose from 50+ chart types including bar, line, pie, area, scatter, "
            "funnel, geographic maps, and KPI widgets. Apply dynamic filters and drill-down "
            "capabilities. Share dashboards or embed them in websites."
        ],
        "response": (
            "Yes, Zoho Analytics can generate natural language insights from your charts. "
            "The AI-powered narrative engine automatically analyzes your data visualizations "
            "and produces written summaries explaining key trends, outliers, and patterns."
        ),
        "label": "hallucinated"
    },
    {
        "id": "H13",
        "query": "Does Zoho Projects support automatic time tracking based on app usage?",
        "retrieved_docs": [
            "Zoho Projects Time Tracking: Log time spent on tasks manually or using the "
            "built-in timer. Start/stop the timer from task detail views. Enter time logs "
            "in bulk using the timesheet view. Set budgeted hours for tasks and projects. "
            "Generate timesheet reports by user, project, or date range."
        ],
        "response": (
            "Yes, Zoho Projects includes automatic time tracking that monitors your application "
            "usage. It detects which project-related applications you are using and automatically "
            "logs time to the corresponding tasks via Settings > Time Tracking > Auto Detection."
        ),
        "label": "hallucinated"
    },

    # === FAITHFUL EXAMPLES (12) ===
    {
        "id": "F01",
        "query": "Does Site24x7 support GitHub repository monitoring?",
        "retrieved_docs": [
            "Site24x7 GitHub Integration: Monitor your GitHub repositories health, "
            "track commit frequency, pull request metrics, and repository activity. "
            "Set up alerts for unusual patterns in your codebase activity. Configure "
            "the integration from Admin > Third Party Integration > GitHub."
        ],
        "response": (
            "Yes, Site24x7 supports GitHub repository monitoring. You can track repository "
            "health, commit frequency, pull request metrics, and overall repository activity. "
            "Set up the integration from Admin > Third Party Integration > GitHub."
        ),
        "label": "faithful"
    },
    {
        "id": "F02",
        "query": "How do I set up multi-language support in Zoho Desk help center?",
        "retrieved_docs": [
            "Zoho Desk Multi-language Support: Zoho Desk supports multiple languages for "
            "your help center portal. Customers can view the help center in their preferred "
            "language. You can create knowledge base articles in multiple languages and "
            "configure language-specific email templates. Supported languages include English, "
            "Spanish, French, German, Japanese, and 40+ others."
        ],
        "response": (
            "You can set up multi-language support by creating knowledge base articles in "
            "different languages and configuring language-specific email templates. Your help "
            "center portal supports 40+ languages allowing customers to view content in their "
            "preferred language."
        ),
        "label": "faithful"
    },
    {
        "id": "F03",
        "query": "Can I handle foreign currency transactions in Zoho Books?",
        "retrieved_docs": [
            "Zoho Books Multi-Currency Support: Handle transactions in multiple currencies "
            "with automatic exchange rate updates. Zoho Books supports 170+ currencies and "
            "automatically fetches real-time exchange rates. Record transactions, generate "
            "invoices, and manage accounts payable/receivable in foreign currencies."
        ],
        "response": (
            "Yes, Zoho Books supports multi-currency transactions with 170+ currencies. "
            "The system automatically fetches real-time exchange rates. You can record "
            "transactions, generate invoices, and manage accounts in foreign currencies "
            "with automatic conversion."
        ),
        "label": "faithful"
    },
    {
        "id": "F04",
        "query": "What task management features are available in Zoho Projects?",
        "retrieved_docs": [
            "Zoho Projects Task Management: Create and organize tasks with customizable "
            "fields, priorities, and deadlines. Assign tasks to team members manually or "
            "in bulk. Use task dependencies to establish project workflows. Track task "
            "progress with Kanban boards, Gantt charts, and custom views."
        ],
        "response": (
            "Zoho Projects offers comprehensive task management including customizable task "
            "fields, priorities, and deadlines. You can assign tasks manually or in bulk, "
            "set up task dependencies for workflows, and track progress using Kanban boards, "
            "Gantt charts, and custom views."
        ),
        "label": "faithful"
    },
    {
        "id": "F05",
        "query": "How does Zoho Analytics handle data imports?",
        "retrieved_docs": [
            "Zoho Analytics Data Import: Import data from various sources including files "
            "(CSV, Excel, JSON), databases (MySQL, PostgreSQL, SQL Server, Oracle), cloud "
            "storage (Google Drive, Dropbox, Box), and business applications. Schedule "
            "automatic data syncs at intervals ranging from every 30 minutes to daily/weekly."
        ],
        "response": (
            "Zoho Analytics supports importing data from files (CSV, Excel, JSON), databases "
            "(MySQL, PostgreSQL, SQL Server, Oracle), cloud storage services, and business "
            "applications. You can schedule automatic data syncs from every 30 minutes to "
            "weekly intervals."
        ),
        "label": "faithful"
    },
    {
        "id": "F06",
        "query": "What are the auto-reply options in Zoho Mail?",
        "retrieved_docs": [
            "Zoho Mail Smart Features: Use canned responses to quickly reply to common "
            "emails with pre-written templates. Create and manage response templates "
            "organized by categories. Set up auto-reply (out-of-office) messages with "
            "customizable schedules. Use email filters and rules to organize incoming mail."
        ],
        "response": (
            "Zoho Mail offers canned responses with pre-written templates organized by "
            "categories for quick replies. You can set up auto-reply out-of-office messages "
            "with customizable schedules. Email filters and rules help automatically organize "
            "incoming mail."
        ),
        "label": "faithful"
    },
    {
        "id": "F07",
        "query": "What IT automation actions can I configure in Site24x7?",
        "retrieved_docs": [
            "Site24x7 IT Automation: Configure automated actions triggered by monitor "
            "alerts. Supported automation actions include restarting Windows services, "
            "executing custom scripts on servers, sending webhook notifications, creating "
            "tickets in ITSM tools, and triggering AWS Lambda functions."
        ],
        "response": (
            "Site24x7 supports several IT automation actions triggered by alerts: restarting "
            "Windows services, executing custom server scripts, sending webhook notifications, "
            "creating ITSM tickets, and triggering AWS Lambda functions."
        ),
        "label": "faithful"
    },
    {
        "id": "F08",
        "query": "Can I access Zoho Creator apps on my phone?",
        "retrieved_docs": [
            "Zoho Creator Mobile Access: Access your Zoho Creator applications on mobile "
            "devices through the Zoho Creator mobile app. View and manage records, submit "
            "forms, and receive push notifications on iOS and Android. Customize the mobile "
            "layout of your forms and reports for a better mobile experience."
        ],
        "response": (
            "Yes, you can access Zoho Creator apps on mobile through the Zoho Creator mobile "
            "app, available on iOS and Android. You can view and manage records, submit forms, "
            "and receive push notifications. The mobile layout can be customized."
        ),
        "label": "faithful"
    },
    {
        "id": "F09",
        "query": "What channels does Zoho Desk support for creating tickets?",
        "retrieved_docs": [
            "Zoho Desk Multi-Channel Ticketing: Create tickets from multiple channels "
            "including email, web forms, social media (Twitter, Facebook), live chat, phone "
            "calls, and community forums. Each channel is automatically mapped to a "
            "department. Phone call tickets are created when agents log call details."
        ],
        "response": (
            "Zoho Desk supports ticket creation from multiple channels: email, web forms, "
            "social media (Twitter, Facebook), live chat, phone calls, and community forums. "
            "Each channel is automatically mapped to a department."
        ),
        "label": "faithful"
    },
    {
        "id": "F10",
        "query": "How do I track expenses in Zoho Books?",
        "retrieved_docs": [
            "Zoho Books Expense Tracking: Record and manage business expenses. Attach "
            "receipts via image upload or email forwarding. Create expense categories and tags "
            "for organization. Set up recurring expenses for regular payments. Generate "
            "expense reports by category, vendor, date range, or project."
        ],
        "response": (
            "You can track expenses in Zoho Books by recording them manually and attaching "
            "receipts via image upload or email forwarding. Organize expenses with categories "
            "and tags, set up recurring expenses, and generate reports by category, vendor, "
            "date, or project."
        ),
        "label": "faithful"
    },
    {
        "id": "F11",
        "query": "How do I log time on tasks in Zoho Projects?",
        "retrieved_docs": [
            "Zoho Projects Time Tracking: Log time spent on tasks manually or using the "
            "built-in timer. Start/stop the timer from task detail views. Enter time logs "
            "in bulk using the timesheet view. Set budgeted hours for tasks and projects. "
            "Generate timesheet reports by user, project, or date range."
        ],
        "response": (
            "You can log time on tasks in Zoho Projects manually or with the built-in timer "
            "from task views. Bulk entry is available via the timesheet view. You can set "
            "budgeted hours and generate timesheet reports by user, project, or date range."
        ),
        "label": "faithful"
    },
    {
        "id": "F12",
        "query": "What visualization options does Zoho Analytics offer?",
        "retrieved_docs": [
            "Zoho Analytics Visualization: Create interactive charts, pivot tables, and "
            "dashboards. Choose from 50+ chart types including bar, line, pie, area, scatter, "
            "funnel, geographic maps, and KPI widgets. Apply dynamic filters and drill-down "
            "capabilities. Share dashboards or embed them in websites."
        ],
        "response": (
            "Zoho Analytics offers 50+ chart types including bar, line, pie, area, scatter, "
            "funnel, geographic maps, and KPI widgets, along with pivot tables and dashboards. "
            "You can apply dynamic filters with drill-down capabilities, share or embed "
            "dashboards."
        ),
        "label": "faithful"
    },
]

# 2. Slice RAGTruth QA subset (30 examples)
ds = load_dataset('wandb/RAGTruth-processed', split='test')
qa_ds = ds.filter(lambda x: x['task_type'] == 'QA')

# Take first 30 examples with clean representation
ragtruth_subset = []
for idx in range(30):
    item = qa_ds[idx]
    lbl_dict = item['hallucination_labels_processed']
    has_hall = lbl_dict.get('evident_conflict', 0) > 0 or lbl_dict.get('baseless_info', 0) > 0
    label_str = "hallucinated" if has_hall else "faithful"
    ragtruth_subset.append({
        "id": f"RT_{idx:02d}",
        "query": item['query'],
        "retrieved_docs": [item['context']],
        "response": item['output'],
        "label": label_str
    })

print(f"Loaded {len(mock_dataset)} mock examples and {len(ragtruth_subset)} RAGTruth QA examples.")


---
## \U0001f528 Section 2: Loading Baseline Tools
We load LettuceDetect and HHEM using our correct weight patch logic.

In [ ]:
# ============================================================
# Loading Baseline Models
# ============================================================
from lettucedetect.models.inference import HallucinationDetector
import transformers
import torch
from transformers import AutoModelForSequenceClassification

# Apply correct weights tying property patch
transformers.PreTrainedModel.all_tied_weights_keys = property(
    lambda self: getattr(self, "_all_tied_weights_keys", {}),
    lambda self, v: setattr(self, "_all_tied_weights_keys", v)
)

print("Loading LettuceDetect...")
lettuce_detector = HallucinationDetector(
    method="transformer",
    model_path="KRLabsOrg/lettucedect-base-modernbert-en-v1",
)

print("Loading HHEM...")
hhem_model = AutoModelForSequenceClassification.from_pretrained(
    "vectara/hallucination_evaluation_model", trust_remote_code=True
).to(DEVICE)
with torch.no_grad():
    hhem_model.t5.transformer.encoder.embed_tokens.weight.data.copy_(
        hhem_model.t5.transformer.shared.weight.data
    )

print("Baselines loaded successfully!")


---
## \U0001f6e1️ Section 3: Loading MiniCheck and Relevance Models
We load MiniCheck and the sentence embedding model.

In [ ]:
# ============================================================
# Load MiniCheck and SentenceTransformers Relevance Model
# ============================================================
from minicheck.minicheck import MiniCheck
from sentence_transformers import SentenceTransformer

print("Loading MiniCheck (Flan-T5-Large)...")
minicheck_scorer = MiniCheck(model_name='flan-t5-large')

print("Loading SentenceTransformer (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

print("Composite evaluation models loaded!")


---
## \U0001f916 Section 4: Local Ollama LLM-as-Judge & Escalation Helper


In [ ]:
# ============================================================
# Local Ollama Helper for LLM-as-Judge and Escalation Tier
# ============================================================
OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'llama3.2:latest'

def query_ollama(prompt, temperature=0.0):
    """Query local Ollama server."""
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "options": {
                    "temperature": temperature
                }
            },
            timeout=30,
        )
        if resp.status_code == 200:
            return resp.json().get("response", "").strip()
    except Exception as e:
        print(f"Ollama error: {e}")
    return None

def ollama_faithfulness_check(query, docs, response):
    """Fallback Ollama LLM-as-Judge check."""
    docs_text = "\n\n".join(docs)
    prompt = (
        "You are a strict fact-checker for a technical support system.\n\n"
        "Below are documents retrieved for a customer's question, and the answer \n"
        "a support bot gave. Determine if the answer is fully supported by the \n"
        "documents.\n\n"
        "Important: a document merely being on the same general topic \n"
        "(e.g., mentioning \"GitHub\") does NOT mean it supports a specific claimed \n"
        "capability (e.g., \"push/pull integration\"). Only mark a claim as \n"
        "supported if the exact capability/fact is explicitly stated in the \n"
        "documents.\n\n"
        "Example of what counts as UNSUPPORTED:\n"
        "Document: \"GitHub integration allows monitoring of repository activity.\"\n"
        "Claim: \"You can push and pull code via the GitHub integration.\"\n"
        "Verdict: UNSUPPORTED \u2014 monitoring is not push/pull capability.\n\n"
        f"Documents:\n{docs_text}\n\n"
        f"Answer:\n{response}\n\n"
        'Respond ONLY with JSON format:\n{"verdict": "faithful" or "hallucinated", "reason": "short explanation"}'
    )
    res = query_ollama(prompt)
    if res:
        try:
            parsed = json.loads(res)
            return parsed.get("verdict", "error")
        except:
            pass
    return "error"


---
## \U0001f6e1️ Section 5: Composite Evaluation Architecture
We implement Relevance Check, MiniCheck scoring, Citation Coverage, Risk Score, and Escalation Tier.

In [ ]:
# ============================================================
# Composite Risk Evaluation Pipeline
# ============================================================
from sentence_transformers import util
from nltk.tokenize import sent_tokenize

RELEVANCE_THRESHOLD = 0.40

def run_composite_eval(query, retrieved_docs, response):
    """Run relevance, MiniCheck faithfulness, citation coverage, risk score, and escalation."""
    # 1. Relevance Check (Max Cosine Similarity)
    q_emb = embedder.encode(query, convert_to_tensor=True)
    doc_embs = embedder.encode(retrieved_docs, convert_to_tensor=True)
    cos_sims = util.cos_sim(q_emb, doc_embs)[0].cpu().tolist()
    relevance_score = max(cos_sims) if cos_sims else 0.0
    is_relevant = relevance_score >= RELEVANCE_THRESHOLD
    
    # 2. Split response into claims/sentences
    claims = sent_tokenize(response)
    if not claims:
        claims = [response]
        
    # 3. MiniCheck Faithfulness & Escalation
    docs_combined = " ".join(retrieved_docs)
    
    # MiniCheck expects doc list to match claim list
    minicheck_docs = [docs_combined] * len(claims)
    res_score = minicheck_scorer.score(docs=minicheck_docs, claims=claims)
    labels, probs = res_score[0], res_score[1]
    
    final_labels = []
    escalated_count = 0
    
    for label, prob, claim in zip(labels, probs, claims):
        # Escalation tier for borderline/uncertain scores
        if 0.30 <= prob <= 0.70:
            verdict = ollama_faithfulness_check(query, retrieved_docs, claim)
            escalated_count += 1
            if verdict == "faithful":
                final_labels.append("faithful")
            else:
                final_labels.append("hallucinated")
        else:
            final_labels.append("faithful" if label == 1 else "hallucinated")
            
    # Faithfulness score is % of faithful claims
    faithful_count = sum(1 for l in final_labels if l == "faithful")
    faithfulness_score = faithful_count / len(claims)
    is_faithful = faithfulness_score >= 1.0  # must be 100% faithful to pass
    
    # 4. Citation Coverage
    cited_docs_count = 0
    for doc in retrieved_docs:
        # Check if this document supports any claims
        doc_docs = [doc] * len(claims)
        doc_res = minicheck_scorer.score(docs=doc_docs, claims=claims)
        doc_labels = doc_res[0]
        if any(l == 1 for l in doc_labels):
            cited_docs_count += 1
            
    citation_coverage = cited_docs_count / len(retrieved_docs) if retrieved_docs else 0.0
    
    # 5. Composite Risk Score
    # Risk is high if context is relevant, but the answer is unfaithful
    high_risk = is_relevant and not is_faithful
    
    return {
        "relevance_score": relevance_score,
        "is_relevant": is_relevant,
        "faithfulness_score": faithfulness_score,
        "is_faithful": is_faithful,
        "citation_coverage": citation_coverage,
        "high_risk": high_risk,
        "escalated_count": escalated_count,
        "predicted_label": "faithful" if is_faithful else "hallucinated"
    }


---
## \U0001f4c3 Section 6: Run Evaluation Loop
We evaluate all tools across both Mock and RAGTruth datasets.

In [ ]:
# ============================================================
# Run Evaluation
# ============================================================
results = {
    "mock": {
        "LettuceDetect": [],
        "HHEM": [],
        "Local-LLM": [],
        "MiniCheck": [],
        "Composite": []
    },
    "ragtruth": {
        "LettuceDetect": [],
        "HHEM": [],
        "Local-LLM": [],
        "MiniCheck": [],
        "Composite": []
    }
}

for dataset_name, dataset in [("mock", mock_dataset), ("ragtruth", ragtruth_subset)]:
    print(f"\nEvaluating dataset: {dataset_name.upper()} ({len(dataset)} examples)")
    print("-" * 50)
    for i, row in enumerate(dataset):
        docs_text = " ".join(row["retrieved_docs"])
        
        # 1. LettuceDetect
        try:
            preds = lettuce_detector.predict(context=row["retrieved_docs"], question=row["query"], answer=row["response"])
            ld_pred = "hallucinated" if any(preds) else "faithful"
        except:
            ld_pred = "error"
        results[dataset_name]["LettuceDetect"].append(ld_pred)
        
        # 2. HHEM
        try:
            hhem_score = float(hhem_model.predict([(docs_text, row["response"])])[0].item())
            hhem_pred = "faithful" if hhem_score > 0.5 else "hallucinated"
        except:
            hhem_pred = "error"
        results[dataset_name]["HHEM"].append(hhem_pred)
        
        # 3. Local-LLM (LLM-as-Judge)
        llm_pred = ollama_faithfulness_check(row["query"], row["retrieved_docs"], row["response"])
        results[dataset_name]["Local-LLM"].append(llm_pred)
        
        # 4. MiniCheck
        try:
            mc_labels, _ = minicheck_scorer.score(docs=[docs_text], claims=[row["response"]])
            mc_pred = "faithful" if mc_labels[0] == 1 else "hallucinated"
        except:
            mc_pred = "error"
        results[dataset_name]["MiniCheck"].append(mc_pred)
        
        # 5. Composite (Our new Multi-Signal Risk pipeline)
        comp = run_composite_eval(row["query"], row["retrieved_docs"], row["response"])
        results[dataset_name]["Composite"].append(comp["predicted_label"])
        
        print(f"  [{row['id']}] GT: {row['label']:12s} | MC: {mc_pred:12s} | Comp: {comp['predicted_label']:12s} | Risk: {'HIGH' if comp['high_risk'] else 'LOW'}")

print("\nEvaluation loop completed successfully!")


---
## \U0001f4ca Section 7: Compute Metrics & Tabulate


In [ ]:
# ============================================================
# Compute Metrics & Tabulate Results
# ============================================================

def compute_metrics(y_true, y_pred, tool_name, dataset_name):
    valid_mask = [(p not in ("error", "unavailable")) for p in y_pred]
    y_true_f = [t for t, v in zip(y_true, valid_mask) if v]
    y_pred_f = [p for p, v in zip(y_pred, valid_mask) if v]

    if not y_true_f:
        return {"tool": tool_name, "dataset": dataset_name,
                "precision": None, "recall": None, "f1": None, "accuracy": None,
                "n_valid": 0, "n_total": len(y_true)}

    y_true_bin = [1 if t == "hallucinated" else 0 for t in y_true_f]
    y_pred_bin = [1 if p == "hallucinated" else 0 for p in y_pred_f]

    return {
        "tool": tool_name,
        "dataset": dataset_name,
        "precision": precision_score(y_true_bin, y_pred_bin, zero_division=0),
        "recall":    recall_score(y_true_bin, y_pred_bin, zero_division=0),
        "f1":        f1_score(y_true_bin, y_pred_bin, zero_division=0),
        "accuracy":  accuracy_score(y_true_bin, y_pred_bin),
        "n_valid":   len(y_true_f),
        "n_total":   len(y_true),
    }

gt_mock = [x["label"] for x in mock_dataset]
gt_rt   = [x["label"] for x in ragtruth_subset]

metrics_rows = []
for dname, gt_list in [("mock", gt_mock), ("ragtruth", gt_rt)]:
    dataset_label = "Mock (Adjacent-Cap.)" if dname == "mock" else "RAGTruth QA Subset"
    for tool_name in ["LettuceDetect", "HHEM", "Local-LLM", "MiniCheck", "Composite"]:
        preds = results[dname][tool_name]
        metrics_rows.append(compute_metrics(gt_list, preds, tool_name, dataset_label))

df_metrics = pd.DataFrame(metrics_rows)

print("=" * 85)
print("📊  COMPOSITE HALLUCINATION EVALUATION METRICS TABLE")
print("=" * 85)
print()

display_df = df_metrics.copy()
for col in ["precision", "recall", "f1", "accuracy"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if x is not None else "N/A")
display_df["coverage"] = display_df.apply(lambda r: f"{r['n_valid']}/{r['n_total']}", axis=1)

print(display_df[["tool", "dataset", "precision", "recall", "f1", "accuracy", "coverage"]].to_string(index=False))


---
## \U0001f4c8 Section 8: Visualizations


In [ ]:
# ============================================================
# Visualizations
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (dname, label) in enumerate([("Mock (Adjacent-Cap.)", "Mock"), ("RAGTruth QA Subset", "RAGTruth")]):
    ax = axes[idx]
    sub_df = df_metrics[df_metrics["dataset"] == dname]
    x = np.arange(len(sub_df))
    w = 0.25
    b1 = ax.bar(x - w, sub_df["precision"], w, label="Precision", color="#2196F3", alpha=0.85)
    b2 = ax.bar(x,     sub_df["recall"],    w, label="Recall",    color="#FF9800", alpha=0.85)
    b3 = ax.bar(x + w, sub_df["f1"],        w, label="F1",        color="#4CAF50", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(sub_df["tool"].values, rotation=25)
    ax.set_ylabel("Score")
    ax.set_title(f"{label} Dataset Comparison", fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("composite_poc_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved to composite_poc_results.png")


---
## \U0001f4dd Section 9: Summary and Recommendations


In [ ]:
# ============================================================
# Final Summary and Recommendations
# ============================================================
print("=" * 80)
print("📝 COMPOSITE POC — FINAL ANALYSIS AND SUMMARY")
print("=" * 80)

# Recommend best models
for dname in ["Mock (Adjacent-Cap.)", "RAGTruth QA Subset"]:
    sub_df = df_metrics[df_metrics["dataset"] == dname]
    best = sub_df.loc[sub_df["f1"].idxmax()]
    print(f"\n🏆 Best on {dname}: {best['tool']} (F1={best['f1']:.3f})")

print("\nAnalysis complete!")
